____
### 1. Imports

In [1]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import random
from dataclasses import dataclass, field

____
### 2. Gym Environment
- simulates a user picking constructing SQL queries using a block editor and provides suggestions for the next "block" given what they have already built
- state : encoding of (blocks placed so far, target query's block sequence)
- action : which block type to suggest next, discrete from the set of blocks available
- reward : 
    - +1 if suggestion matches what a correct build path would do next,
    - partial credit of +0.5 via similarity 
    - -1 for an invalid/contradictory move
    - terminal bonus when the full query is correctly assembled.
 


#### 2.1 Defining Block Types
- Block types indicate the SQL keyword to be used
- Therefore the block types indicate the action space of the environment and makes the action space discrete

In [2]:
BLOCK_TYPES = [
    "SELECT",
    "SELECT_COLUMN",
    "FROM",
    "JOIN", # for connecting 2 tables on certain conditions
    "JOIN_CONDITION",   # ON clause
    "WHERE", # for filtering 
    "WHERE_CONDITION",
    "GROUP_BY",
    "ORDER_BY",
    "LIMIT",
    "END",
]

In [3]:
BLOCK_TO_IDX = {b: i for i, b in enumerate(BLOCK_TYPES)} 
# creates a dictionary with block : index 
#enumerate creates a list with tuples of (index, block)

IDX_TO_BLOCK = {i: b for b, i in BLOCK_TO_IDX.items()} 
# creates a dictionary with index : block 
# BLOCK_TO_IDX.items() creates a list with tuples of (block, index)

N_BLOCKS = len(BLOCK_TYPES) #total number of types of blocks

MAX_SEQUENCE_LEN = 10  # cap so state vector has fixed size

#### 2.2 Defining Question Types
- The problem bank is a set of hardcoded canonical patterns used byt he blocks defined inside `BLOCK_TYPES` 
- Each problem is the correct block-type sequence a learner should build
- In the real app these would be derived from actual exercise definitions

In [4]:
PROBLEM_BANK = [
    ["SELECT", "SELECT_COLUMN", "FROM", "END"],
    ["SELECT", "SELECT_COLUMN", "FROM", "WHERE", "WHERE_CONDITION", "END"],
    ["SELECT", "SELECT_COLUMN", "FROM", "GROUP_BY", "END"],
    ["SELECT", "SELECT_COLUMN", "FROM", "WHERE", "WHERE_CONDITION", "GROUP_BY", "ORDER_BY", "END"],
    ["SELECT", "SELECT_COLUMN", "FROM", "ORDER_BY", "LIMIT", "END"],

    # single join
    ["SELECT", "SELECT_COLUMN", "FROM", "JOIN", "JOIN_CONDITION", "END"],

    # join + filter
    ["SELECT", "SELECT_COLUMN", "FROM", "JOIN", "JOIN_CONDITION",
     "WHERE", "WHERE_CONDITION", "END"],

    # multiple joins (two tables joined in)
    ["SELECT", "SELECT_COLUMN", "FROM", "JOIN", "JOIN_CONDITION",
     "JOIN", "JOIN_CONDITION", "END"],

    # join + group/order, the "kitchen sink" case
    ["SELECT", "SELECT_COLUMN", "FROM", "JOIN", "JOIN_CONDITION",
     "WHERE", "WHERE_CONDITION", "GROUP_BY", "ORDER_BY", "END"],
]

#### 2.3 Creating class to represent the episode state
- simple data class used to store episode data 

In [5]:
@dataclass
class EpisodeState:
    target_seq: list = field(default_factory=list)
    built_seq: list = field(default_factory=list)

- `target_seq` : storing the answer key of the given question sampled from `PROBLEM_BANK`
- `build_seq` : what the agent has correctly placed and built, adding blocks after the agent adds a block
- both characteristics are typed as a `list`
    - `field(default_factory = list)` is used to give each instance a fresh empty list to avoid mutable lists used across instances

#### 2.4 Creating Environment 

In [6]:
class SQLEnv(gym.Env):
    metadata = {"render_modes": ["human"]} # for methods to see what render modes are supported
    def __init__(self, render_mode=None):
        super().__init__()
        self.render_mode = render_mode
        obs_dim = MAX_SEQUENCE_LEN + 1 + N_BLOCKS
        self.observation_space = spaces.Box(
            low=0, high=max(N_BLOCKS, MAX_SEQUENCE_LEN),
            shape=(obs_dim,), dtype=np.float32,
        )
        self.action_space = spaces.Discrete(N_BLOCKS)
        self.state: EpisodeState | None = None

    def required_blocks_vector(self):
        remaining = self.state.target_seq[len(self.state.built_seq):]
        vec = np.zeros(N_BLOCKS, dtype=np.float32)
        for b in remaining:
            vec[BLOCK_TO_IDX[b]] = 1.0
        return vec

    def get_obs(self):
        built_idx = [BLOCK_TO_IDX[b] + 1 for b in self.state.built_seq]  
        built_idx = built_idx[:MAX_SEQUENCE_LEN]
        built_idx += [0] * (MAX_SEQUENCE_LEN - len(built_idx))
        step_count = np.array([len(self.state.built_seq) / MAX_SEQUENCE_LEN], dtype=np.float32)
        required = self.required_blocks_vector()
        obs = np.concatenate([
            np.array(built_idx, dtype=np.float32),
            step_count,
            required
        ])
        return obs

    def get_info(self):
        return {"built_seq": list(self.state.built_seq),
            "target_seq": list(self.state.target_seq)
            }

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        target = random.choice(PROBLEM_BANK)
        self.state = EpisodeState(target_seq=list(target), built_seq=[])
        return self.get_obs(), self.get_info()

    def step(self, action: int):
        block = IDX_TO_BLOCK[action]
        target = self.state.target_seq
        pos = len(self.state.built_seq)
        terminated = False
        truncated = False
        reward = 0.0
        expected_block = target[pos] if pos < len(target) else "END"
        if block == expected_block:
            self.state.built_seq.append(block)
            if block == "END":
                reward = 5.0 
                terminated = True
            else:
                reward = 1.0
        else:
            reward = -1.0
            if pos >= MAX_SEQUENCE_LEN - 1:
                truncated = True
        if len(self.state.built_seq) >= MAX_SEQUENCE_LEN:
            truncated = True
        return self.get_obs(), reward, terminated, truncated, self.get_info()

    def render(self):
        if self.render_mode == "human":
            print(f"Built so far: {self.state.built_seq}")
            print(f"Target:       {self.state.target_seq}")

##### 2.41 Initialisation 
```python
def __init__(self, render_mode=None):
    super().__init__()
    self.render_mode = render_mode
    obs_dim = MAX_SEQUENCE_LEN + 1 + N_BLOCKS
    self.observation_space = spaces.Box(
        low=0, high=max(N_BLOCKS, MAX_SEQUENCE_LEN),
        shape=(obs_dim,), dtype=np.float32,
    )
    self.action_space = spaces.Discrete(N_BLOCKS)
    self.state: EpisodeState | None = None
```

- `obs_dim = MAX_SEQUENCE_LEN + 1 + N_BLOCKS` : observation dimension
    - given that `MAX_SEQUENCE_LEN = 10` and `N_BLOCKS = 11`, the observation vector is 22 numbers concatencated
    - the first `MAX_SEQUENCE_LEN` numbers is referring to the sequence of blocks already built by the user 
    - the next number `+1` is the step count
    - the last `N_BLOCKS` numbers are an array with the number of each block at each index used
- `self.observation_space = spaces.Box(low=0, high=max(N_BLOCKS, MAX_SEQUENCE_LEN), shape=(obs_dim,), dtype=np.float32)` : a vector representing the observation space
    - `low=0` forces a floor of 0 all values inside the array 
    - `high=max(N_BLOCKS, MAX_SEQUENCE_LEN)` forces a ceiling for all values, the higher value between `MAX_SEQUENCE_LEN` and `N_BLOCKS`
    - `shape=(obs_dim,)` : the shape of the array is equal to a vector of dimension = `obs_dim`
- `self.action_space = spaces.Discrete(N_BLOCKS)` : a singular number that represents the next block to be placed 
    - `spaces.Discrete(N_BLOCKS)` : creates a discrete space with a number representing each block 
- `self.state: EpisodeState | None = None` : initialises an empty `EpisodeState` object

##### 2.42 `required_blocks_vector()` - helper to construct the vector to represents the blocks required

```python
def required_blocks_vector(self):
    remaining = self.state.target_seq[len(self.state.built_seq):]
    vec = np.zeros(N_BLOCKS, dtype=np.float32)
    for b in remaining:
        vec[BLOCK_TO_IDX[b]] = 1.0
    return vec
```
- `remaining = self.state.target_seq[len(self.state.built_seq):]` : finds the blocks that still need to appear
    - `self.state.target_seq` returns the target that is being built towards
    - `[len(self.state.built_seq):]` indexes the target from the end of the built seq to the end of the target
- `vec = np.zeros(N_BLOCKS, dtype=np.float32)` : creates an array of zeroes of length equal to `N_BLOCKS`\
- `for b in remaining` : iterate through each value inside the remaining
- `vec[BLOCK_TO_IDX[b]] = 1.0` : if the block is required, changes the 0 to a 1


##### 2.43 `get_obs()` - to obtain the observation from the state

```python
def get_obs(self):
    built_idx = [BLOCK_TO_IDX[b] + 1 for b in self.state.built_seq] 
    built_idx = built_idx[:MAX_SEQUENCE_LEN]
    built_idx += [0] * (MAX_SEQUENCE_LEN - len(built_idx))
    step_count = np.array([len(self.state.built_seq) / MAX_SEQUENCE_LEN], dtype=np.float32)
    required = self.required_blocks_vector()
    obs = np.concatenate([
        np.array(built_idx, dtype=np.float32),
        step_count,
        required
        ])
    return obs
```
- `built_idx = [BLOCK_TO_IDX[b] + 1 for b in self.state.built_seq] ` : to build the first `MAX_SEQUENCE_LEN` numbers in the observation array 
    - `for b in self.state.built_seq` accesses the current `built_seq` inside `self.state`
    - `BLOCK_TO_IDX[b] + 1` finds the block inside the `BLOCK_TO_IDX` dictionary, `+1` is to make it such that 0 corresponds to an empty slot
- `built_idx = built_idx[:MAX_SEQUENCE_LEN]` : to slice off only the last `MAX_SEQUENCE_LEN` numbers 
-  `built_idx += [0] * (MAX_SEQUENCE_LEN - len(built_idx))` : pads the `built_idx` arrays with zeroes to make it up to size
- `step_count = np.array([len(self.state.built_seq) / MAX_SEQUENCE_LEN], dtype=np.float32)` : calculates the completion of the sequence
    - `len(self.state.built_seq) / MAX_SEQUENCE_LEN` calculates how complete the current query is compared to the max possible query 
    - ie, if `built_seq length = 3` and `MAX_SEQUENCE_LEN = 10`, this computes 0.3 and shows that its 30% through
- `required = self.required_blocks_vector()` : builds the remaining blocks that are required by calling the helper `required_blocks_vector()`
- `obs = np.concatenate([np.array(built_idx, dtype=np.float32), step_count,required])`: combines the 3 arrays into 1 long numpy array 

##### 2.44 `get_info()` - helper to return state info
- returns the state info as a dictionary, as required by the standar methods of Gymnasium's `reset()` and `step()` methods

```python
def get_info(self):
    return {"built_seq": list(self.state.built_seq),
        "target_seq": list(self.state.target_seq)
        }
```

##### 2.45 `reset()` - to reset the environment and clear all states, observation and targets

```python
def reset(self, seed=None, options=None):
    super().reset(seed=seed)
    target = random.choice(PROBLEM_BANK)
    self.state = EpisodeState(target_seq=list(target), built_seq=[])
    return self.get_obs(), self.get_info()
```

- `target = random.choice(PROBLEM_BANK)` : chooses a random problem from `PROBLEM_BANK`
- `self.state = EpisodeState(target_seq=list(target), built_seq=[])` : resets `self.state`
    - `target_seq=list(target)` to make the target the new problem chosen
    - `built_seq=[]` to make the built sequence an empty array 

##### 2.46 `step()` - to progress the environment according to the action input 

```python
def step(self, action: int):
    block = IDX_TO_BLOCK[action]
    target = self.state.target_seq
    pos = len(self.state.built_seq)
    terminated = False
    truncated = False
    reward = 0.0
    expected_block = target[pos] if pos < len(target) else "END"
    if block == expected_block:
        self.state.built_seq.append(block)
        if block == "END":
            reward = 5.0 
            terminated = True
        else:
            reward = 1.0
    else:
        reward = -1.0
        if pos >= MAX_SEQUENCE_LEN - 1:
            truncated = True
    if len(self.state.built_seq) >= MAX_SEQUENCE_LEN:
        truncated = True
    return self.get_obs(), reward, terminated, truncated, self.get_info()
```
- `block = IDX_TO_BLOCK[action]` : converts the action from the index to the actual block 
- `target = self.state.target_seq` : accesses the target from `self.state`
- `pos = len(self.state.built_seq)` : finds the current length of what is already built, `state.built_seq`
- `terminated = False` , `truncated = False`, `reward = 0.0` : initialising empty values 
- `expected_block = target[pos] if pos < len(target) else "END"` : checks the expected block at the position
    - `expected_block` at the given position according to the `target` 
    - only uses the `expected_block` if the position is not yet at the end, else it is automatically assigned to `"END"`
- `if block == expected_block` : if the block is correct 
    - `self.state.built_seq.append(block)` adds the block to the `built_seq` under `self.state`
    - `if block == "END"` : if the block is correct and is `"END`, provides a bonus reward of `reward = 5.0 ` and terminates the episode with `terminated = True`
    - else `reward = 1.0` for providing the correct block 
- `else` : if the block is wrong
    - `reward = -1.0` loses reward for choosing the wrong block 
    - `if pos >= MAX_SEQUENCE_LEN - 1 : truncated = True` : guardrail to check if the sequence is too long
- `if len(self.state.built_seq) >= MAX_SEQUENCE_LEN: truncated = True` : checks if the sequence is at max length after adding the correct block and terminates the episode

#### 2.5 Exploring the environment 

In [7]:
env = SQLEnv(render_mode="human")
obs, info = env.reset(seed=0)
env.render()

total_reward = 0.0
for t in range(15):
    action = env.action_space.sample()  # random sampling
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    print(f"step {t}: suggested={IDX_TO_BLOCK[action]:<16} reward={reward:+.1f}")
    if terminated or truncated:
        print("Episode finished:", "terminated" if terminated else "truncated")
        break

print(f"\nTotal reward: {total_reward:.1f}")
env.render()


Built so far: []
Target:       ['SELECT', 'SELECT_COLUMN', 'FROM', 'JOIN', 'JOIN_CONDITION', 'WHERE', 'WHERE_CONDITION', 'GROUP_BY', 'ORDER_BY', 'END']
step 0: suggested=LIMIT            reward=-1.0
step 1: suggested=SELECT_COLUMN    reward=-1.0
step 2: suggested=JOIN             reward=-1.0
step 3: suggested=LIMIT            reward=-1.0
step 4: suggested=JOIN_CONDITION   reward=-1.0
step 5: suggested=JOIN             reward=-1.0
step 6: suggested=WHERE_CONDITION  reward=-1.0
step 7: suggested=JOIN_CONDITION   reward=-1.0
step 8: suggested=SELECT           reward=+1.0
step 9: suggested=LIMIT            reward=-1.0
step 10: suggested=FROM             reward=-1.0
step 11: suggested=SELECT           reward=-1.0
step 12: suggested=LIMIT            reward=-1.0
step 13: suggested=SELECT_COLUMN    reward=+1.0
step 14: suggested=SELECT           reward=-1.0

Total reward: -11.0
Built so far: ['SELECT', 'SELECT_COLUMN']
Target:       ['SELECT', 'SELECT_COLUMN', 'FROM', 'JOIN', 'JOIN_CONDITION',

### 3. Training the DQN model
- a Deep-Q-Network model is used in this case since the action space is discrete
- The classes are imported from another one of my repositories : https://github.com/vruaaan/reinforcement-learning-projects

#### 3.1 Importing classes from outside 

In [8]:
from agents import Agent
from buffers import ReplayBuffer
from dqn import DQNAgent

#### 3.2 Training and evaluating the agent

In [9]:
env = SQLEnv()
agent = DQNAgent(env=env, name="sql-dqn")

In [10]:
agent.train(total_timesteps=500_000, learning_starts=200, log_every=10)
rewards = agent.evaluate(n_episodes=100)

Timestep     688 | Episode   10 | Reward:   -31.00 | Epsilon: 0.987
Timestep    1389 | Episode   20 | Reward:   -63.00 | Epsilon: 0.974
Timestep    2017 | Episode   30 | Reward:   -39.00 | Epsilon: 0.962
Timestep    2571 | Episode   40 | Reward:   -53.00 | Epsilon: 0.951
Timestep    3044 | Episode   50 | Reward:   -39.00 | Epsilon: 0.942
Timestep    3583 | Episode   60 | Reward:   -47.00 | Epsilon: 0.932
Timestep    4086 | Episode   70 | Reward:   -46.00 | Epsilon: 0.922
Timestep    4451 | Episode   80 | Reward:   -28.00 | Epsilon: 0.915
Timestep    4874 | Episode   90 | Reward:   -24.00 | Epsilon: 0.907
Timestep    5302 | Episode  100 | Reward:   -56.00 | Epsilon: 0.899
Timestep    5645 | Episode  110 | Reward:   -15.00 | Epsilon: 0.893
Timestep    5970 | Episode  120 | Reward:   -13.00 | Epsilon: 0.887
Timestep    6298 | Episode  130 | Reward:    -9.00 | Epsilon: 0.880
Timestep    6706 | Episode  140 | Reward:    -8.00 | Epsilon: 0.873
Timestep    7026 | Episode  150 | Reward:   -24.

#### 3.3 Explanation of evaluation results
From the results of 100 evaluation episodes:
```md
Mean reward: 10.97 +/- 1.80
Reward range: [8.00, 14.00]
Mean steps: 6.97
```

Environment:`SQLEnv` — 9 problems in `PROBLEM_BANK`, reward = +1 per correct block, +5 terminal bonus for correct `END`, -1 for a wrong suggestion.
| Metric              | Random baseline | **DQN (obtained)** | Optimal (theoretical) |
|---------------------|:---------------:|:-------------------:|:----------------------:|
| Mean reward         | -15.12           | **10.97 ± 1.8**    | 10.78                  |
| Reward range        | [-20.0, 5.0]     | **[8.00, 14.00]**   | [8.00, 14.00]          |
| Mean steps          | —                | **6.97**             | 6.78                   |
| % of optimal        | —                | **~119%**           | 100%                   |

- the obtained reward range exactly matches the theoretical optimal range, meaning the agent made zero wrong-block suggestions across the eval episodes.
- ~0.19 gap between obtained (10.97) and theoretical (10.78) mean reward is attributable to sampling variance from evaluating on only 20 episodes rather than a policy deficiency — the DQN agent has effectively solved this environment.

#### 3.4 Saving trained model to `.pth` file

In [11]:
agent.save("basicmodel.pth")

'basicmodel.pth'